#Unity Catalog Setup
##Creates catalog, schemas, and Delta tables for the Wyckoff pipeline.

In [0]:
%run ../config/config

In [0]:
%sql
GRANT USE CATALOG ON CATALOG `wyckoff_catalog` TO `khangthesis@ikhang2501gmail.onmicrosoft.com`

In [0]:
%sql
GRANT ALL PRIVILEGES ON CATALOG `wyckoff_catalog` TO `khangthesis@ikhang2501gmail.onmicrosoft.com`

In [0]:
%sql
ALTER CATALOG `wyckoff_catalog` OWNER TO `khangthesis@ikhang2501gmail.onmicrosoft.com`

In [0]:
%sql
SELECT current_user();

#### 1. Create external location

In [0]:
spark.sql(f"""
    CREATE EXTERNAL LOCATION IF NOT EXISTS wyckoff_external
    URL '{external_location}/'
    WITH (STORAGE CREDENTIAL {storage_credential})
    COMMENT 'External storage location for Wyckoff pipeline data'
""")
print(f"External location successfull created at: {external_location}/")


#### 1.1 Verify external location

In [0]:
%sql
DESCRIBE EXTERNAL LOCATION wyckoff_external;

#### 2. Create catalog

In [0]:
spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {catalog}
    MANAGED LOCATION '{catalog_managed_location}'
""")
spark.sql(f"USE CATALOG {catalog}")
print(f"Catalog '{catalog}' successfull created at: {catalog_managed_location}")

#### 3. Create schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS wyckoff_catalog.bronze;
CREATE SCHEMA IF NOT EXISTS wyckoff_catalog.silver;
CREATE SCHEMA IF NOT EXISTS wyckoff_catalog.gold;

###4. Create Delta Tables

#####4.1 Create Bronze

In [0]:
%sql
CREATE TABLE IF NOT EXISTS wyckoff_catalog.bronze.raw_ohlcv (
    symbol       STRING      NOT NULL COMMENT 'Stock ticker symbol',
    date         DATE        NOT NULL COMMENT 'Trading date',
    open         DOUBLE                COMMENT 'Opening price',
    high         DOUBLE                COMMENT 'Highest price',
    low          DOUBLE                COMMENT 'Lowest price',
    close        DOUBLE                COMMENT 'Closing price',
    volume       BIGINT                COMMENT 'Trading volume',
    exchange     STRING                COMMENT 'Exchange: HOSE, HNX, UPCOM',
    ingested_at  TIMESTAMP             COMMENT 'Ingestion timestamp',
    source       STRING                COMMENT 'Data source'
)
USING DELTA
COMMENT 'Raw OHLCV data from Vietnamese stock market'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);

#####4.2 Optimize after Bronze ingestion

In [0]:
%sql
OPTIMIZE wyckoff_catalog.bronze.raw_ohlcv ZORDER BY (symbol, date);

##### 4.3 Create Silver: cleaned_ohlcv

In [0]:
%sql
CREATE TABLE IF NOT EXISTS wyckoff_catalog.silver.cleaned_ohlcv (
    symbol          STRING      NOT NULL,
    date            DATE        NOT NULL,
    exchange        STRING,
    open            DOUBLE,
    high            DOUBLE,
    low             DOUBLE,
    close           DOUBLE,
    volume          BIGINT,
    prev_close      DOUBLE      COMMENT 'Previous day close',
    daily_return    DOUBLE      COMMENT 'Close-to-close return',
    log_return      DOUBLE      COMMENT 'Log return',
    typical_price   DOUBLE      COMMENT '(H + L + C) / 3',
    spread          DOUBLE      COMMENT 'High - Low',
    price_limit_hit BOOLEAN     COMMENT 'True / False'
)
USING DELTA
COMMENT 'Cleaned and validated OHLCV with derived columns'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
);

##### 4.4 Optimize after Silver ingestion: cleaned_ohlcv

In [0]:
%sql
OPTIMIZE wyckoff_catalog.silver.cleaned_ohlcv ZORDER BY (symbol, date);

##### 4.5 Create Silver: technical_indicators

In [0]:
%sql
CREATE TABLE IF NOT EXISTS wyckoff_catalog.silver.technical_indicators (
    symbol      STRING NOT NULL,
    date        DATE   NOT NULL,
    rsi_14      DOUBLE,
    rsi_28      DOUBLE,
    macd        DOUBLE,
    macd_signal DOUBLE,
    macd_hist   DOUBLE,
    ema_12      DOUBLE,
    ema_26      DOUBLE,
    sma_20      DOUBLE,
    sma_50      DOUBLE,
    bb_upper    DOUBLE,
    bb_middle   DOUBLE,
    bb_lower    DOUBLE,
    bb_width    DOUBLE,
    bb_pct      DOUBLE,
    obv         DOUBLE,
    vwap_20     DOUBLE,
    volume_ratio DOUBLE,
    atr_14      DOUBLE,
    adx_14      DOUBLE,
    plus_di     DOUBLE,
    minus_di    DOUBLE,
    stoch_k     DOUBLE,
    stoch_d     DOUBLE
)
USING DELTA
COMMENT 'Technical indicators computed from cleaned OHLCV'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact'   = 'true'
);

##### 4.6 Optimize after Silver transformation: technical_indicators

In [0]:
%sql
OPTIMIZE wyckoff_catalog.silver.technical_indicators ZORDER BY (symbol, date);

##### 4.7 Create Silver: wyckoff_features

In [0]:
%sql
CREATE TABLE IF NOT EXISTS wyckoff_catalog.silver.wyckoff_features (
    symbol              STRING      NOT NULL,
    date                DATE        NOT NULL,
    price_spread        DOUBLE      COMMENT 'Normalized high-low spread',
    close_position      DOUBLE      COMMENT 'Close position within bar (0-1)',
    volume_climax_up    BOOLEAN     COMMENT 'Volume climax on up bar',
    volume_climax_down  BOOLEAN     COMMENT 'Volume climax on down bar',
    stopping_volume     BOOLEAN     COMMENT 'High volume narrow spread',
    no_demand           BOOLEAN     COMMENT 'Low volume narrow up bar',
    no_supply           BOOLEAN     COMMENT 'Low volume narrow down bar',
    spring_signal       BOOLEAN     COMMENT 'Break below support with recovery',
    upthrust_signal     BOOLEAN     COMMENT 'Break above resistance with failure',
    sos_signal          BOOLEAN     COMMENT 'Sign of strength',
    sow_signal          BOOLEAN     COMMENT 'Sign of weakness',
    effort_vs_result    DOUBLE      COMMENT 'Volume vs price move ratio',
    test_signal         BOOLEAN     COMMENT 'Test of support/resistance on low volume',
    support_level       DOUBLE      COMMENT 'Rolling support (min low)',
    resistance_level    DOUBLE      COMMENT 'Rolling resistance (max high)',
    trading_range_width DOUBLE      COMMENT 'Resistance - Support',
    trend_direction     INT         COMMENT '1=up, -1=down, 0=neutral (SMA20 vs SMA50)',
    days_in_range           DOUBLE      COMMENT 'Consecutive days price within S/R band',
    volume_trend            DOUBLE      COMMENT 'Volume SMA momentum (10-bar pct change)',
    relative_position       DOUBLE      COMMENT 'Close position within trading range [0,1]',
    breakout_strength       DOUBLE      COMMENT '(close - resistance) / ATR',
    volume_price_divergence DOUBLE      COMMENT 'Price-volume alignment signal'
)
USING DELTA
COMMENT 'Wyckoff-specific features for phase detection'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);

##### 4.8 Optimize after Silver transformation: wyckoff_features

In [0]:
%sql
OPTIMIZE wyckoff_catalog.silver.wyckoff_features ZORDER BY (symbol, date);

##### 4.9 Create Gold: phase_labels

In [0]:
%sql
CREATE TABLE IF NOT EXISTS wyckoff_catalog.gold.phase_labels (
    symbol              STRING      NOT NULL,
    date                DATE        NOT NULL,
    phase_label         INT         NOT NULL    COMMENT 'Wyckoff phase (0-9)',
    phase_name          STRING                  COMMENT 'Phase name',
    phase_confidence    DOUBLE                  COMMENT 'Labeling confidence (0-1)',
    tr_support          DOUBLE                  COMMENT 'Trading range support level',
    tr_resistance       DOUBLE                  COMMENT 'Trading range resistance level',
    tr_id               INT                     COMMENT 'Trading range identifier'
)
USING DELTA
COMMENT 'Wyckoff phase labels'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);

##### 4.10 Optimize after Gold labeling: phase_labels

In [0]:
%sql
OPTIMIZE wyckoff_catalog.gold.phase_labels ZORDER BY (symbol, date);

##### 4.11 Create Gold: ml_features

In [0]:
%sql
CREATE TABLE IF NOT EXISTS wyckoff_catalog.gold.ml_features (
    symbol          STRING      NOT NULL,
    date            DATE        NOT NULL    COMMENT 'Trading date',
    exchange        STRING                  COMMENT 'Exchange code (HSX, HNX, UPCOM)',
    split           STRING                  COMMENT 'train/val/test',
    phase_label     INT         NOT NULL    COMMENT 'Wyckoff phase label (0-9)',
    price_limit_hit DOUBLE                  COMMENT 'Whether day hit price limit',
    daily_return    DOUBLE                  COMMENT 'Daily return'
)
USING DELTA
COMMENT 'ML-ready flat per-day features with labels (windowing done at training time)'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);

##### 4.12 Optimize after Gold features: ml_features

In [0]:
%sql
OPTIMIZE wyckoff_catalog.gold.ml_features ZORDER BY (symbol, date);

##### 4.13 Create Gold: model_metrics

In [0]:
%sql
CREATE TABLE IF NOT EXISTS wyckoff_catalog.gold.model_metrics (
    model_name          STRING      NOT NULL,
    run_id              STRING,
    eval_date           TIMESTAMP,
    split               STRING      COMMENT 'val or test',
    accuracy            DOUBLE,
    macro_f1            DOUBLE,
    weighted_f1         DOUBLE,
    per_phase_f1        MAP<STRING, DOUBLE>     COMMENT 'F1 per phase name',
    per_phase_precision MAP<STRING, DOUBLE>,
    per_phase_recall    MAP<STRING, DOUBLE>,
    confusion_matrix    ARRAY<ARRAY<INT>>       COMMENT '10x10 confusion matrix'
)
USING DELTA
COMMENT 'Model evaluation metrics for comparison'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);

##### 4.14 Create Gold: cnn_lstm_predictions

In [0]:
%sql
CREATE TABLE IF NOT EXISTS wyckoff_catalog.gold.cnn_lstm_predictions (
    symbol          STRING          NOT NULL,
    date            DATE            NOT NULL,
    model_name      STRING                      COMMENT 'Model that produced prediction',
    run_id          STRING                      COMMENT 'MLflow parent run ID of the ensemble',
    predicted       INT             NOT NULL    COMMENT 'Predicted phase label (0-9)',
    actual          INT                         COMMENT 'Ground-truth phase label (0-9)',
    macro_f1        DOUBLE                      COMMENT 'Model test macro F1 (broadcast onto every row)',
    weighted_f1     DOUBLE                      COMMENT 'Model test weighted F1',
    accuracy        DOUBLE                      COMMENT 'Model test accuracy',
    probabilities   ARRAY<DOUBLE>               COMMENT 'Softmax probabilities for all 10 phases'
)
USING DELTA
COMMENT 'Per-row CNN-LSTM predictions on the test set, with ensemble-averaged probabilities'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);

##### 4.15 Create Gold: event_study_results

In [0]:
%sql
CREATE TABLE IF NOT EXISTS wyckoff_catalog.gold.event_study_results (
    phase           STRING      NOT NULL    COMMENT 'Predicted phase name',
    n_signals       BIGINT                  COMMENT 'Number of predictions of this phase in the test set',
    mean_5d_pct     DOUBLE                  COMMENT 'Mean forward return over 5 days (%)',
    tstat_5d        DOUBLE                  COMMENT 'One-sample t-stat vs H0: mean = 0',
    win_5d_pct      DOUBLE                  COMMENT 'Share of signals with positive 5d return (%)',
    mean_10d_pct    DOUBLE,
    tstat_10d       DOUBLE,
    win_10d_pct     DOUBLE,
    mean_20d_pct    DOUBLE,
    tstat_20d       DOUBLE,
    win_20d_pct     DOUBLE,
    mean_60d_pct    DOUBLE,
    tstat_60d       DOUBLE,
    win_60d_pct     DOUBLE
)
USING DELTA
COMMENT 'Event-study forward-return statistics per predicted Wyckoff phase'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);

##### 4.16 Create Gold: component_contribution_results

In [0]:
%sql
CREATE TABLE IF NOT EXISTS wyckoff_catalog.gold.component_contribution_results (
    variant         STRING      NOT NULL    COMMENT 'Ablation variant: full, no_cnn, no_lstm, no_attn',
    use_cnn         BOOLEAN                 COMMENT 'MultiScale CNN branch enabled',
    use_bilstm      BOOLEAN                 COMMENT 'BiLSTM block enabled',
    use_attention   BOOLEAN                 COMMENT 'Multi-Head Attention + residual/LayerNorm enabled',
    seed            INT                     COMMENT 'Training seed',
    macro_f1        DOUBLE                  COMMENT 'Test macro F1',
    weighted_f1     DOUBLE                  COMMENT 'Test weighted F1',
    accuracy        DOUBLE                  COMMENT 'Test accuracy',
    val_f1          DOUBLE                  COMMENT 'Best validation macro F1 during training',
    params_count    BIGINT                  COMMENT 'Total trainable parameters for this variant',
    eval_date       TIMESTAMP               COMMENT 'Evaluation timestamp',
    run_id          STRING                  COMMENT 'MLflow nested run ID for reproducibility'
)
USING DELTA
COMMENT 'Per-variant metrics from the CNN-LSTM architecture ablation study'
TBLPROPERTIES (
    'delta.autoOptimize.optimizeWrite' = 'true',
    'delta.autoOptimize.autoCompact' = 'true'
);

### 5. Verify Setup

In [0]:
%sql
SHOW TABLES IN wyckoff_catalog.bronze;

In [0]:
%sql
SHOW TABLES IN wyckoff_catalog.silver;

In [0]:
%sql
SHOW TABLES IN wyckoff_catalog.gold;

### 6. Describe Key Tables

In [0]:
%sql
DESCRIBE TABLE wyckoff_catalog.bronze.raw_ohlcv;

In [0]:
%sql
DESCRIBE TABLE wyckoff_catalog.gold.phase_labels;

In [0]:
%sql
DESCRIBE TABLE wyckoff_catalog.gold.ml_features;

### 7. Verify External Storage Location

In [0]:
%sql
DESCRIBE CATALOG EXTENDED wyckoff_catalog;

In [0]:
%sql
DESCRIBE DETAIL wyckoff_catalog.bronze.raw_ohlcv;